# Generate Experiment 3 Graphs

This notebook grabs the collated results file from storage and generates graphs
comparing model performance on the ICWM task

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick
import shutil
import os
import constants

In [38]:
FIGURES_PATH = constants.ROOT_DIR + "/ICWM/Figures"

### Manually Inspect Model Results


In [31]:
import pickle as pkl
import shutil

STORAGE_PATH = constants.ROOT_DIR + "/ICWM/3rd_party_results/claude-haiku-4-5-20251001/thinking_-1/world_size_16/dimension_2/no_information_two_step_vertical_rule/results.pkl"
if os.path.exists("./results.pkl"):
  os.remove("./results.pkl")
shutil.copy(
    STORAGE_PATH,
    os.path.join(".", "results.pkl"),
)

results = pkl.load(open("./results.pkl", "rb"))
print(results['answer'][0])
print(results['prompt'][0][-200:])
print(results['response'][0][:15])
print(results['grid'][0])


# Generate NTP Results

In [53]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp3_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [54]:
ntp_results = results_df[results_df["prompt"] == "next_token_prediction"]

In [55]:
ntp_results["Setting"] = "Dim: " + ntp_results["dimension"].astype(str) + " World: " + ntp_results["world_size"].astype(str) 

prompt_map = {
    "Dim: 1 World: 16": 1,
    "Dim: 1 World: 25": 2,
    "Dim: 2 World: 16": 3,
    "Dim: 2 World: 25": 4,
    }

models_map = {
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "openai/gpt-5": "GPT 5",
    "openai/gpt-5-mini": "GPT 5 Mini",
    "claude-haiku-4-5-20251001": "Claude Haiku 4.5",
    }

ntp_results["Complexity"] = ntp_results["Setting"].map(prompt_map)
ntp_results["Model"] = ntp_results["model"].map(models_map)
ntp_results["Jittered Complexity"] = np.random.normal(ntp_results["Complexity"], 0.1)
ntp_results.head(5)

In [56]:
# Make NTP
sns.set_theme(style='darkgrid')
ntp_results = ntp_results[ntp_results["thinking"].isin([-1, 5000])]
ax = sns.scatterplot(data=ntp_results, x="Jittered Complexity", y="acc", hue="Model", marker="o", s=70)
ax.set_xticks(ntp_results['Complexity'])

# Optional: Rotate labels for better readability
plt.xticks(rotation=40)
plt.ylabel("Accuracy")
plt.xlabel("Topology")
plt.ylim(0, 1.05)
plt.title("Next Token Prediction: Frontier Models", fontsize=14)
ax.set_xticklabels(ntp_results['Setting'])

plt.savefig("./Exp3_NTP.pdf", format="pdf", bbox_inches="tight", dpi=600)
if os.path.exists(os.path.join(FIGURES_PATH, "Exp3_NTP.pdf")):
  os.remove(os.path.join(FIGURES_PATH, "Exp3_NTP.pdf"))
shutil.copy("./Exp3_NTP.pdf", os.path.join(FIGURES_PATH, "Exp3_NTP.pdf"))

# Generate ICWM Results

In [62]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp3_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [64]:
rule_map = {
    "one_step_vertical_rule": 1,
    "two_step_vertical_rule": 2,
    "three_step_knight_rule": 3,
    }
xlabel_map = {
    "one_step_vertical_rule": "One Step",
    "two_step_vertical_rule": "Two Step",
    "three_step_knight_rule": "Three Step",
    }
icwm_results = results_df[results_df["prompt"].str.contains("no_information")]
icwm_results = icwm_results[icwm_results["thinking"].isin([-1, 5000])]
icwm_results = icwm_results[icwm_results["model"].isin(["gemini-2.5-pro", "gemini-2.5-flash", "openai/gpt-5", "openai/gpt-5-mini"])]
icwm_results["Rules"] = icwm_results["rule"].map(rule_map)
icwm_results["xlabel"] = icwm_results["rule"].map(xlabel_map)
icwm_results["Jittered Rules"] = np.random.normal(icwm_results["Rules"], 0.1)

In [65]:
# Make ICWM Accuracy Graph
icwm_results["Topology"] = "Dim: " + icwm_results["dimension"].astype(str) + " World: " + icwm_results["world_size"].astype(str)
ax = sns.scatterplot(data=icwm_results, x="Jittered Rules", y="acc", hue="Topology", s=80)
ax.set_xticks(icwm_results['Rules'])

# Set the labels using the values from the other column
ax.set_xticklabels(icwm_results['xlabel'])

# Optional: Rotate labels for better readability
plt.xticks(rotation=40)
plt.ylabel("Accuracy")
plt.xlabel("Rule")

plt.ylim(0, 1)
plt.title("In-Context World Modeling: Frontier Models", fontsize=16)

plt.savefig("./Exp3_ICWM_Accuracy.pdf", format="pdf", bbox_inches="tight", dpi=600)
if os.path.exists(os.path.join(FIGURES_PATH, "Exp3_ICWM_Accuracy.pdf")):
  os.remove(os.path.join(FIGURES_PATH, "Exp3_ICWM_Accuracy.pdf"))
shutil.copy("./Exp3_ICWM_Accuracy.pdf", os.path.join(FIGURES_PATH, "Exp3_ICWM_Accuracy.pdf"))

# Generate Thinking vs. No Thinking Comparison

### NTP

In [93]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp3_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [100]:
ntp_results["Setting"] = "Dim: " + ntp_results["dimension"].astype(str) + " World: " + ntp_results["world_size"].astype(str) 

prompt_map = {
    "Dim: 1 World: 16": 1,
    "Dim: 1 World: 25": 2,
    "Dim: 2 World: 16": 3,
    "Dim: 2 World: 25": 4,
    }

models_map = {
    "gemini-2.5-flash": "Gemini 2.5 Flash",
    "gemini-2.5-pro": "Gemini 2.5 Pro",
    "openai/gpt-5": "GPT 5",
    "openai/gpt-5-mini": "GPT 5 Mini",
    "claude-haiku-4-5-20251001": "Claude Haiku 4.5",
    }

ntp_results["Complexity"] = ntp_results["Setting"].map(prompt_map)
ntp_results["Model"] = ntp_results["model"].map(models_map)
ntp_results["Thinking"] = np.where(ntp_results['thinking'] == 5000, 'Thinking', 'Limited')

ntp_results["Jittered Complexity"] = np.random.normal(ntp_results["Complexity"], 0.1)
ntp_results.head(5)

In [101]:
# Make NTP
sns.set_theme(style='darkgrid')
flash_results = ntp_results[ntp_results["Model"].isin(["Gemini 2.5 Flash"])]
ax = sns.barplot(data=flash_results, x="Complexity", y="acc", hue="Thinking")
ax.set_xticks(flash_results['Complexity'] - 1)

# Optional: Rotate labels for better readability
plt.xticks(rotation=40)
plt.ylabel("Accuracy")
plt.xlabel("Topology")
plt.ylim(0, 1.05)
plt.title("Next Token Prediction: Gemini Flash Thinking vs. Not", fontsize=14)
ax.set_xticklabels(flash_results['Setting'])

plt.savefig("./Exp3_Flash_NTP_Thinking_Comparison.pdf", format="pdf", bbox_inches="tight", dpi=600)
if gfile.Exists(os.path.join(FIGURES_PATH, "Exp3_Flash_NTP_Thinking_Comparison.pdf")):
  gfile.Remove(os.path.join(FIGURES_PATH, "Exp3_Flash_NTP_Thinking_Comparison.pdf"))
gfile.Copy("./Exp3_Flash_NTP_Thinking_Comparison.pdf", os.path.join(FIGURES_PATH, "Exp3_Flash_NTP_Thinking_Comparison.pdf"))

In [102]:
# Make NTP
sns.set_theme(style='darkgrid')
pro_results = ntp_results[ntp_results["Model"].isin(["Gemini 2.5 Pro"])]
ax = sns.barplot(data=pro_results, x="Complexity", y="acc", hue="Thinking")
ax.set_xticks(pro_results['Complexity'] - 1)

# Optional: Rotate labels for better readability
plt.xticks(rotation=40)
plt.ylabel("Accuracy")
plt.xlabel("Topology")
plt.ylim(0, 1.05)
plt.title("Next Token Prediction: Gemini Pro Thinking vs. Not", fontsize=14)
ax.set_xticklabels(pro_results['Setting'])

plt.savefig("./Exp3_Pro_NTP_Thinking_Comparison.pdf", format="pdf", bbox_inches="tight", dpi=600)
if gfile.Exists(os.path.join(FIGURES_PATH, "Exp3_Pro_NTP_Thinking_Comparison.pdf")):
  gfile.Remove(os.path.join(FIGURES_PATH, "Exp3_Pro_NTP_Thinking_Comparison.pdf"))
gfile.Copy("./Exp3_Pro_NTP_Thinking_Comparison.pdf", os.path.join(FIGURES_PATH, "Exp3_Pro_NTP_Thinking_Comparison.pdf"))

### ICWM

In [108]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp3_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [114]:
rule_map = {
    "one_step_vertical_rule": 1,
    "two_step_vertical_rule": 2,
    "three_step_knight_rule": 3,
    }
xlabel_map = {
    "one_step_vertical_rule": "One Step",
    "two_step_vertical_rule": "Two Step",
    "three_step_knight_rule": "Three Step",
    }
icwm_results = results_df[results_df["prompt"].str.contains("no_information")]
icwm_results["Rules"] = icwm_results["rule"].map(rule_map)
icwm_results["xlabel"] = icwm_results["rule"].map(xlabel_map)
icwm_results["Jittered Rules"] = np.random.normal(icwm_results["Rules"], 0.075)
icwm_results["Thinking"] = np.where(icwm_results['thinking'] == 5000, 'Thinking', 'Limited')


In [115]:
# Make ICWM Accuracy Graph
flash_results = icwm_results[icwm_results["model"].isin(["gemini-2.5-flash"])]
flash_results["Topology"] = "Dim: " + flash_results["dimension"].astype(str) + " World: " + flash_results["world_size"].astype(str)
ax = sns.scatterplot(data=flash_results, x="Jittered Rules", y="acc", hue="Thinking", s=80)
ax.set_xticks(flash_results['Rules'])

# Set the labels using the values from the other column
ax.set_xticklabels(flash_results['xlabel'])

# Optional: Rotate labels for better readability
plt.xticks(rotation=40)
plt.ylabel("Accuracy")
plt.xlabel("Rule")

plt.ylim(0, 1)
plt.title("Gemini Flash ICWM: Thinking vs. Not", fontsize=16)

plt.savefig("./Exp3_Flash_Thinking_ICWM.pdf", format="pdf", bbox_inches="tight", dpi=600)
if gfile.Exists(os.path.join(FIGURES_PATH, "Exp3_Flash_Thinking_ICWM.pdf")):
  gfile.Remove(os.path.join(FIGURES_PATH, "Exp3_Flash_Thinking_ICWM.pdf"))
gfile.Copy("./Exp3_Flash_Thinking_ICWM.pdf", os.path.join(FIGURES_PATH, "Exp3_Flash_Thinking_ICWM.pdf"))


In [110]:
# Make ICWM Accuracy Graph
pro_results = icwm_results[icwm_results["model"].isin(["gemini-2.5-pro"])]
pro_results["Topology"] = "Dim: " + pro_results["dimension"].astype(str) + " World: " + pro_results["world_size"].astype(str)
ax = sns.scatterplot(data=pro_results, x="Jittered Rules", y="acc", hue="Thinking", s=80)
ax.set_xticks(pro_results['Rules'])

# Set the labels using the values from the other column
ax.set_xticklabels(pro_results['xlabel'])

# Optional: Rotate labels for better readability
plt.xticks(rotation=40)
plt.ylabel("Accuracy")
plt.xlabel("Rule")

plt.ylim(0, 1)
plt.title("Gemini Pro ICWM: Thinking vs. Not", fontsize=16)

plt.savefig("./Exp3_Pro_Thinking_ICWM.pdf", format="pdf", bbox_inches="tight", dpi=600)
if gfile.Exists(os.path.join(FIGURES_PATH, "Exp3_Pro_Thinking_ICWM.pdf")):
  gfile.Remove(os.path.join(FIGURES_PATH, "Exp3_Pro_Thinking_ICWM.pdf"))
gfile.Copy("./Exp3_Pro_Thinking_ICWM.pdf", os.path.join(FIGURES_PATH, "Exp3_Pro_Thinking_ICWM.pdf"))


# Generate Explicit Grid Results

In [120]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp3_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [121]:
rule_map = {
    "one_step_vertical_rule": 1,
    "two_step_vertical_rule": 2,
    "three_step_knight_rule": 3,
    }
xlabel_map = {
    "one_step_vertical_rule": "One Step",
    "two_step_vertical_rule": "Two Step",
    "three_step_knight_rule": "Three Step",
    }
models_map = {
        "gemini-2.5-flash": "Gemini 2.5 Flash",
        "gemini-2.5-pro": "Gemini 2.5 Pro"
}
explicit_results = results_df[results_df["prompt"].str.contains("explicit_grid")]
explicit_results["Rules"] = explicit_results["rule"].map(rule_map)
explicit_results["xlabel"] = explicit_results["rule"].map(xlabel_map)
explicit_results["Model"] = explicit_results["model"].map(models_map)
explicit_results["Jittered Rules"] = np.random.normal(explicit_results["Rules"], 0.075)
explicit_results["Thinking"] = np.where(explicit_results['thinking'] == 5000, 'Thinking', 'Limited')


In [125]:
# Make Explicit Accuracy Graph
ax = sns.barplot(data=explicit_results, x="Model", y="acc", hue="Thinking")
sns.move_legend(ax, "lower left")

plt.ylabel("Accuracy")

plt.ylim(0, 1)
plt.title("Gemini Explicit Grid Results", fontsize=16)

plt.savefig("./Exp3_Explicit.pdf", format="pdf", bbox_inches="tight", dpi=600)
if gfile.Exists(os.path.join(FIGURES_PATH, "Exp3_Explicit.pdf")):
  gfile.Remove(os.path.join(FIGURES_PATH, "Exp3_Explicit.pdf"))
gfile.Copy("./Exp3_Explicit.pdf", os.path.join(FIGURES_PATH, "Exp3_Explicit.pdf"))


# Generate Hints Analysis

In [126]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp3_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [129]:
rule_map = {
    "one_step_vertical_rule": 1,
    "two_step_vertical_rule": 2,
    "three_step_knight_rule": 3,
    }
xlabel_map = {
    "one_step_vertical_rule": "One Step",
    "two_step_vertical_rule": "Two Step",
    "three_step_knight_rule": "Three Step",
    }
models_map = {
        "gemini-2.5-flash": "Gemini 2.5 Flash",
        "gemini-2.5-pro": "Gemini 2.5 Pro"
}

prompt_map = {
    "high_information": "Hint",
    "no_information": "No Hint",
}
hint_results = results_df[results_df["prompt"].str.contains("information")]
hint_results["Rules"] = hint_results["rule"].map(rule_map)
hint_results["xlabel"] = hint_results["rule"].map(xlabel_map)
hint_results["Prompt"] = hint_results["prompt"].map(prompt_map)
hint_results["Model"] = hint_results["model"].map(models_map)
hint_results["Thinking"] = np.where(hint_results['thinking'] == 5000, '', 'Limited')
hint_results["Model"] = hint_results["Model"] + " " + hint_results["Thinking"] + " Thinking"


In [132]:
# Make Hint Accuracy Graph
ax = sns.barplot(data=hint_results, x="Model", y="acc", hue="Prompt")
sns.move_legend(ax, "lower left")

plt.ylabel("Accuracy")
plt.xticks(rotation=40)

plt.ylim(0, 1)
plt.title("Gemini Hint Results", fontsize=16)

plt.savefig("./Exp3_Hint.pdf", format="pdf", bbox_inches="tight", dpi=600)
if gfile.Exists(os.path.join(FIGURES_PATH, "Exp3_Hint.pdf")):
  gfile.Remove(os.path.join(FIGURES_PATH, "Exp3_Hint.pdf"))
gfile.Copy("./Exp3_Hint.pdf", os.path.join(FIGURES_PATH, "Exp3_Hint.pdf"))


# @ToDo Generate Construct Grid Analysis

In [133]:
import shutil

STORAGE_OPEN_RESULTS_PATH = constants.ROOT_DIR + "/ICWM/results/collated_exp3_grid_results.csv"

if os.path.exists("./results.csv"):
  os.remove("./results.csv")
shutil.copy(STORAGE_OPEN_RESULTS_PATH, "./results.csv")
results_df = pd.read_csv("./results.csv")  


In [ ]:
models_map = {
        "gemini-2.5-flash": "Gemini 2.5 Flash",
        "gemini-2.5-pro": "Gemini 2.5 Pro"
}


struct_results = results_df[results_df["prompt"].str.contains("construct")]
struct_results["Model"] = struct_results["model"].map(models_map)
struct_results["Thinking"] = np.where(struct_results['thinking'] == 5000, '', 'Limited')
struct_results["World Size"] = struct_results["world_size"]
struct_results["Model"] = struct_results["Model"] + " " + struct_results["Thinking"] + " Thinking"


In [ ]:
# Make structure description Accuracy Graph
ax = sns.barplot(data=struct_results, x="World Size", y="acc", hue="Model")
sns.move_legend(ax, "lower left")

plt.ylabel("Accuracy")
plt.xticks(rotation=40)

plt.ylim(0, 1)
plt.title("Gemini Structure Description Results", fontsize=16)

plt.savefig("./Exp3_struct.pdf", format="pdf", bbox_inches="tight", dpi=600)
if gfile.Exists(os.path.join(FIGURES_PATH, "Exp3_struct.pdf")):
  gfile.Remove(os.path.join(FIGURES_PATH, "Exp3_struct.pdf"))
gfile.Copy("./Exp3_struct.pdf", os.path.join(FIGURES_PATH, "Exp3_struct.pdf"))
